# CS5296 Local One-Click Demo Notebook

This notebook is for **local reproducibility + live demo recording**.

- It runs a fast, reliable **live path** (`facebook_tiny`) end-to-end.
- It generates real outputs: raw JSONL, summary CSV, and latency plot.
- It also shows your report-ready full results for presentation.


## 0) Usage
Run cells from top to bottom.

1. run preflight + service startup
2. run live benchmark cells
3. show generated table/plot and key findings


In [ ]:
# ---- Config ----
from pathlib import Path

# Set this if your notebook is NOT opened from repo root
REPO_DIR = Path('.').resolve()

# Quick live demo settings (fast)
LIVE_DATASET_NAME = 'facebook_tiny'
LIVE_WORKLOAD = 'benchmarks/facebook_tiny.json'
LIVE_WARMUP = 1
LIVE_MEASURED = 3

# Output tags for this notebook run
TAG = 'live-demo'

print('REPO_DIR =', REPO_DIR)

In [ ]:
# ---- Helpers ----
import os, shlex, subprocess, sys

def run(cmd, check=True, cwd=None):
    print('\n$ ' + cmd)
    p = subprocess.run(cmd, shell=True, cwd=str(cwd or REPO_DIR), text=True,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f'Command failed ({p.returncode}): {cmd}')
    return p

def which(cmd):
    p = subprocess.run(f'command -v {shlex.quote(cmd)}', shell=True, text=True,
                      stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    return p.stdout.strip() if p.returncode == 0 else None

def detect_compose_cmd():
    p = subprocess.run('docker compose version', shell=True, text=True,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if p.returncode == 0:
        return 'docker compose'
    if which('docker-compose'):
        return 'docker-compose'
    return None


In [ ]:
# ---- 1) Preflight ----
import platform

print('Python:', sys.version)
print('Platform:', platform.platform())

if sys.version_info < (3, 12):
    print('WARNING: Project requires Python >= 3.12 (pyproject.toml).')

docker_bin = which('docker')
print('docker:', docker_bin or 'NOT FOUND')

compose_cmd = detect_compose_cmd()
print('compose command:', compose_cmd or 'NOT FOUND')

if not docker_bin or not compose_cmd:
    raise RuntimeError('Docker/Compose not ready. Install Docker and retry.')

# Docker daemon check (auto-heal for macOS + colima)
p = subprocess.run('docker ps', shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(p.stdout)

if p.returncode != 0 and which('colima'):
    print('Docker daemon not ready. Trying: colima start ...')
    run('colima start --cpu 4 --memory 8 --disk 60', check=False)
    run('docker context use colima', check=False)
    p = subprocess.run('docker ps', shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)

if p.returncode != 0:
    print('Docker daemon still unavailable. You can either:')
    print('1) retry colima start')
    print('2) open Docker Desktop, then rerun this cell')
    raise RuntimeError('Docker daemon not running.')

print('Preflight OK')


In [ ]:
# ---- 2) Install Python package (editable) ----
from pathlib import Path
REPO_DIR = Path('/Users/bujuexiao/Documents/CityU/semB/CS5296/project/cs5296-graph-multihop-optimization-main')
print(REPO_DIR)



run('python3 -m pip install -e ".[dev]"')
!pwd && ls pyproject.toml


In [ ]:
# ---- 3) Use mirror images, then retag to official names expected by compose ----
run("docker pull docker.m.daocloud.io/library/postgres:16")
run("docker tag docker.m.daocloud.io/library/postgres:16 postgres:16")

run("docker pull docker.m.daocloud.io/library/neo4j:5.20")
run("docker tag docker.m.daocloud.io/library/neo4j:5.20 neo4j:5.20")

compose_cmd = detect_compose_cmd()
run(f"{compose_cmd} -f infra/docker-compose.yml up -d --pull never neo4j postgres")
run(f"{compose_cmd} -f infra/docker-compose.yml ps")


In [ ]:
# ---- 4) Prepare tiny dataset + load backends ----
run('python3 scripts/prepare_dataset.py --input tests/fixtures/raw/facebook_tiny.txt --dataset-name facebook_tiny --symmetrize --output-dir datasets/derived/facebook_tiny')
run('python3 scripts/load_neo4j.py --dataset-dir datasets/derived/facebook_tiny')
run('python3 scripts/load_postgres.py --dataset-dir datasets/derived/facebook_tiny')

In [ ]:
# ---- 5) Smoke test (live artifact run) ----
run('python3 scripts/smoke_test_core_backends.py')

In [ ]:
# ---- 6) Run live benchmark on tiny dataset ----
neo_raw = f'results/raw/neo4j-{LIVE_DATASET_NAME}-{TAG}.jsonl'
pg_raw  = f'results/raw/postgres-{LIVE_DATASET_NAME}-{TAG}.jsonl'

run(f'python3 scripts/run_benchmark.py --backend neo4j --dataset-name {LIVE_DATASET_NAME} --workload-file {LIVE_WORKLOAD} --warmup-count {LIVE_WARMUP} --measured-count {LIVE_MEASURED} --output {neo_raw}')
run(f'python3 scripts/run_benchmark.py --backend postgres --dataset-name {LIVE_DATASET_NAME} --workload-file {LIVE_WORKLOAD} --warmup-count {LIVE_WARMUP} --measured-count {LIVE_MEASURED} --output {pg_raw}')

print('Raw outputs:', neo_raw, pg_raw)

In [ ]:
# ---- 7) Aggregate + plot live outputs (fixed paths) ----
import pandas as pd
from pathlib import Path

base = Path(REPO_DIR)

neo_csv = base / f"results/summary/neo4j-{LIVE_DATASET_NAME}-{TAG}.csv"
pg_csv  = base / f"results/summary/postgres-{LIVE_DATASET_NAME}-{TAG}.csv"
cmp_csv = base / f"results/summary/{LIVE_DATASET_NAME}-{TAG}-comparison.csv"
cmp_png = base / f"results/summary/{LIVE_DATASET_NAME}-{TAG}-p50-latency.png"

run(f"python3 scripts/aggregate_results.py --input results/raw/neo4j-{LIVE_DATASET_NAME}-{TAG}.jsonl --output {neo_csv}", cwd=base)
run(f"python3 scripts/aggregate_results.py --input results/raw/postgres-{LIVE_DATASET_NAME}-{TAG}.jsonl --output {pg_csv}", cwd=base)

df = pd.concat([pd.read_csv(neo_csv), pd.read_csv(pg_csv)], ignore_index=True)
df.to_csv(cmp_csv, index=False)

run(f"python3 scripts/plot_results.py --input {cmp_csv} --output {cmp_png}", cwd=base)

print("Summary outputs:", neo_csv, pg_csv, cmp_csv, cmp_png)
print("Exists:", neo_csv.exists(), pg_csv.exists(), cmp_csv.exists(), cmp_png.exists())


In [ ]:
# ---- 8) Show live measurement table and plot ----
import pandas as pd
from pathlib import Path
from IPython.display import display, Image

base = Path(REPO_DIR)
cmp_csv = base / f"results/summary/{LIVE_DATASET_NAME}-{TAG}-comparison.csv"
cmp_png = base / f"results/summary/{LIVE_DATASET_NAME}-{TAG}-p50-latency.png"

live_df = pd.read_csv(cmp_csv)
display(live_df[['backend','dataset','query_type','p50_latency_ms','p95_latency_ms','p99_latency_ms']])

display(Image(filename=str(cmp_png)))


In [ ]:
# ---- 9) Show report-ready full results (for final presentation) ----
import pandas as pd
from pathlib import Path
from IPython.display import display, Image

base = Path(REPO_DIR)

fb_csv = base / "results/summary/facebook-full-comparison.csv"
tw_csv = base / "results/summary/twitter-top10000-comparison.csv"
fb_png = base / "results/summary/facebook-full-p50-latency.png"
tw_png = base / "results/summary/twitter-top10000-p50-latency.png"

print("facebook_full files:", fb_csv.exists(), fb_png.exists())
print("twitter_top10000 files:", tw_csv.exists(), tw_png.exists())

fb = pd.read_csv(fb_csv)
tw = pd.read_csv(tw_csv)

print('facebook_full')
display(fb[['backend','query_type','p50_latency_ms','p95_latency_ms','p99_latency_ms']])
display(Image(filename=str(fb_png)))

print('twitter_top10000')
display(tw[['backend','query_type','p50_latency_ms','p95_latency_ms','p99_latency_ms']])
display(Image(filename=str(tw_png)))


## Demo Script Hint 
- We first show a live artifact run on a tiny dataset for reproducibility and command correctness.
- Then we present report-scale benchmark evidence on facebook_full and twitter_top10000.
- Key finding 1: facebook_full favors PostgreSQL + closure_3.
- Key finding 2: twitter_top10000 favors Neo4j on neighbors/common_neighbors, while PostgreSQL remains strongest on shortest_path.
- Caveat: this is PostgreSQL + closure_3 precomputation versus Neo4j online traversal.


In [ ]:
# ---- 10) Optional cleanup ----
# compose_cmd = detect_compose_cmd()
# run(f"{compose_cmd} -f infra/docker-compose.yml down")